# Hivemind on LongMemEval (search only)

This notebook measures Hivemind's search on all 470 answerable LongMemEval_S questions (the 30 abstention questions are left out) with Qwen3-Embedding-0.6B, then on a 20-per-type sample (about 110 questions) with Qwen3-Embedding-8B.

**Before you run it:** in the right-hand panel, set *Accelerator* to **GPU T4 x2** and switch **Internet** on. Kaggle asks for phone verification once before it allows either.

**To run:** click **Save Version**, choose **Save & Run All (Commit)**, then **Save**. It runs in the background, likely for 4 to 7 hours, and you can close the tab. Set `REF` in the first code cell to measure a branch or tag other than `main`. When it finishes, open the version, go to **Output**, download `longmemeval-qwen3-0.6b.md`, `longmemeval-qwen3-8b.md` and the two `.json` files, and keep them with the measurement write-up.

The 0.6B results are saved before 8B starts, so they survive even if the 8B part fails.

If a run stops partway in an interactive session, run the last cell again: vectors already computed are kept in `/tmp/lme` and are not redone.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q uv
REF = "main"  # the branch or tag to measure
!rm -rf /tmp/hivemind && git clone -q --depth 1 -b {REF} https://github.com/monochrome-muncher/hivemind /tmp/hivemind
!cp /tmp/hivemind/scripts/longmemeval-gpu/lme_gpu.py /tmp/lme_gpu.py
!cd /tmp/hivemind && uv sync -q && uv run python -c "import tests.eval.longmemeval; print('harness ok')"
!mkdir -p /tmp/lme && curl -sSL -o /tmp/lme/longmemeval_s_cleaned.json https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json && ls -la /tmp/lme
import transformers, torch
print("transformers", transformers.__version__, "torch", torch.__version__, "GPUs", torch.cuda.device_count())

In [ ]:
!python /tmp/lme_gpu.py '[{"tag": "qwen3-0.6b", "model": "Qwen/Qwen3-Embedding-0.6B", "dim": 1024, "batch_tokens": 32768}, {"tag": "qwen3-8b", "model": "Qwen/Qwen3-Embedding-8B", "dim": 2000, "per_type": 20, "batch_tokens": 16384}]'